# O plano de execução do relatório

## Demanda da aula 4

Lembra do feed de eventos que a operação anunciou na aula 1? Ele chegou: uma linha para cada **lance** de cada
partida, com o minuto, a seleção e o tipo do lance (passe, disputa de bola, falta, chute, corte, escanteio e gol).

Com o feed, a operação montou um job que gera um relatório simples para a transmissão: **quantos lances de cada tipo
aconteceram em cada Copa**. O relatório sai certo, mas demora, e com o feed de todas as Copas chegando a cada rodada
ele vai estourar o prazo. O recado veio direto: descubram por que o job é lento e deixem ele rápido.

Nesta aula, você faz esse diagnóstico como se faz no dia a dia: lê o plano de execução, encontra a operação cara,
ajusta, mede de novo e registra o antes e o depois.

| Etapa | O que fazer | Notebook |
|---|---|---|
| Task 1 | Ler o plano de execução do relatório e medir quanto ele demora hoje | `01_explain` |
| Task 2 | Encontrar o shuffle e ajustar o número de partições | `02_shuffle_e_particoes` |
| Task 3 | Remover a configuração herdada com broadcast e AQE, reaproveitar resultado com cache e registrar o antes e o depois | `03_cache_broadcast_aqe` |
| Task 4 | Investigar um job travado numa task só (data skew) com a Spark UI | `04_skew_e_spark_ui` |

**Pronto quando:** o plano do relatório está lido e explicado, cada ajuste tem a medição de antes e depois registrada
em `docs/otimizacao.md`, e você sabe reconhecer e resolver um job preso numa task só.

## Task 1: ler o plano de execução do relatório e medir quanto ele demora hoje

In [ ]:
# a sessão Spark deste notebook: já vem pronta (é o código do vídeo 1.4, com outro appName)
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-plano-de-execucao")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

## O feed de eventos

O feed está em **Parquet**: um formato que guarda o schema junto com o dado (aula 5).

1,6 milhão de lances contra 562 partidas: uma tabela **grande** e uma **pequena**.

## O que o Spark faz antes de executar

Quando a ação chega, o Spark enxerga a receita inteira (lazy evaluation, aula 2), o otimizador (**Catalyst**) reescreve
o plano, e o **plano físico** decide **como** executar. O `explain()` mostra o plano físico.

## Lendo um plano simples

A árvore se lê **de baixo para cima**.

| No código | No plano |
|---|---|
| `spark.read.parquet(EVENTOS)` | `Scan parquet` |
| `.filter(F.col("event_type") == "Goal")` | `Filter` |
| `.select("match_id", "minute", "team")` | `Project` |

- **`ReadSchema`**: só as colunas usadas são lidas (**column pruning**).
- **`PushedFilters`**: o filtro vai para a leitura (**predicate pushdown**).

## O job da operação

O job da operação roda com duas configurações herdadas de um job antigo: o **AQE** desligado e o **broadcast join
automático** desligado (vídeo 4.3).

O relatório, numa função: cada chamada monta um DataFrame novo, com a configuração daquele momento.

Duas leituras embaixo (feed e partidas), o join no meio e a contagem em cima.

| Nó | O que significa |
|---|---|
| `FileScan` | leitura de um arquivo |
| `Filter`, `Project` | filtrar linhas; escolher ou calcular colunas |
| `HashAggregate` | uma agregação (aparece em dupla) |
| **`Exchange`** | **troca de dados entre partições** (shuffle) |
| `SortMergeJoin` | join que junta, pela chave, as linhas dos dois lados |

Três `Exchange`: o feed pela chave do join, as partidas pelo `dropDuplicates` e a contagem por ano e tipo.

## Quanto o job demora hoje

`%timeit` é um comando do Jupyter: executa a linha algumas vezes (`-n1 -r3`: 3 vezes) e mostra o tempo médio.

## Resultado da task 1

Plano lido (column pruning, um `SortMergeJoin` e três `Exchange`) e tempo atual medido. Próxima task: o shuffle.

## Para consultar depois

- **Modos do `explain`:** `explain()` mostra o plano físico; `explain(True)` mostra também os planos lógicos, antes e
  depois do otimizador; `explain(mode="formatted")` separa a árvore dos detalhes.
- **A ordem não importa:** filtrar antes ou depois do `select` gera o mesmo plano, porque o otimizador reescreve a
  consulta.
- **Em SQL:** `spark.sql("EXPLAIN SELECT ...")` mostra o plano de uma consulta SQL.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()